In [14]:
import numpy as np

In [9]:
def angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b

    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(cos_angle))

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision

BaseOptions = mp.tasks.BaseOptions

MODEL_PATH = "pose_landmarker_full.task"
VIDEO_PATH = "./videos/incorrect_1.mp4"

options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.VIDEO,
)

landmarker = vision.PoseLandmarker.create_from_options(options)

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_idx = 0

# Желаемая высота
target_height = 480
target_aspect_ratio = 9 / 16
target_width = int(target_height * target_aspect_ratio)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Ресайз до 9:16
    frame_resized = cv2.resize(frame, (target_width, target_height))

    timestamp_ms = int((frame_idx / fps) * 1000)
    frame_idx += 1

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
    )

    result = landmarker.detect_for_video(mp_image, timestamp_ms)


    RED = (0, 0, 255)
    GREEN = (0, 255, 0)
    
    HIP_2 = 24
    HIP = 23
    KNEE = 25
    ANKLE = 27
    SHOULDER = 11

    if result.pose_landmarks:
        for pose in result.pose_landmarks:
            landmarks = pose
            for lm_i in range(len(pose) - 1):
                lm = pose[lm_i]
                x = int(lm.x * frame_resized.shape[1])
                y = int(lm.y * frame_resized.shape[0])
                hip = landmarks[HIP]
                knee = landmarks[KNEE]
                ankle = landmarks[ANKLE]
                shoulder = landmarks[SHOULDER]
                print(angle((hip.x, hip.y), (knee.x, knee.y), (ankle.x, ankle.y)))
                if lm_i == HIP or lm_i == HIP_2:
                    color = GREEN if angle((hip.x, hip.y), (knee.x, knee.y), (ankle.x, ankle.y)) > 130 else RED
                    cv2.circle(frame_resized, (x, y), 5, color, -1)
                else:
                    cv2.circle(frame_resized, (x, y), 5, GREEN, -1)

    cv2.imshow("Pose Tracking", frame_resized)

    if cv2.waitKey(int(1000 / fps)) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.7780342956308
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.8250543250917
178.825054

In [47]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision

BaseOptions = mp.tasks.BaseOptions

MODEL_PATH = "pose_landmarker_full.task"
VIDEO_PATH = "./videos/incorrect_2.mp4"
OUTPUT_PATH = "./videos/output_incorrect_2.mp4"

options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.VIDEO,
)

landmarker = vision.PoseLandmarker.create_from_options(options)

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_idx = 0

# 9:16 480p
target_height = 960
target_aspect_ratio = 9 / 16
target_width = int(target_height * target_aspect_ratio)

# ВАЖНО: VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (target_width, target_height)
)

RED = (0, 0, 255)
GREEN = (0, 255, 0)

HIP_2 = 24
HIP = 23
KNEE = 25
ANKLE = 27
SHOULDER = 11

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_resized = cv2.resize(frame, (target_width, target_height))

    timestamp_ms = int((frame_idx / fps) * 1000)
    frame_idx += 1

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
    )

    result = landmarker.detect_for_video(mp_image, timestamp_ms)

    if result.pose_landmarks:
        for pose in result.pose_landmarks:
            landmarks = pose

            for lm_i in range(len(pose) - 1):
                lm = pose[lm_i]

                x = int(lm.x * frame_resized.shape[1])
                y = int(lm.y * frame_resized.shape[0])

                hip = landmarks[HIP]
                knee = landmarks[KNEE]
                ankle = landmarks[ANKLE]

                ang = angle(
                    (hip.x, hip.y),
                    (knee.x, knee.y),
                    (ankle.x, ankle.y)
                )

                if lm_i == HIP or lm_i == HIP_2:
                    color = GREEN if ang > 70 else RED
                    cv2.circle(frame_resized, (x, y), 5, color, -1)
                else:
                    cv2.circle(frame_resized, (x, y), 5, GREEN, -1)

    # 👉 СЮДА СОХРАНЕНИЕ КАДРА
    out.write(frame_resized)

    cv2.imshow("Pose Tracking", frame_resized)

    if cv2.waitKey(int(1000 / fps)) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()